# VeriCall — Execution-Verified Post-Training for Tool Calling
### Reproducible Training & Evaluation Pipeline (Colab T4 / L4 / A100 GPU)
**Author:** Bhuvnesh Sahu ([@bhuvneshsahu01](https://github.com/bhuvneshsahu01))

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhuvneshsahu01/vericall/blob/main/VeriCall_Colab_Pro.ipynb)

---
### Dual-Engine Architecture
- **Engine A (Fast):** Unsloth 4-bit fast kernel patching.
- **Engine B (Crash-Proof Fallback):** HuggingFace `transformers` + `peft` + `bitsandbytes` (NF4 4-bit).
Runs seamlessly on any environment without dependency or directory path crashes.


### Stage 1: Environment Setup & Dependencies


In [ ]:
# Install dependencies with unsloth-zoo and standard HF stack
!pip install --quiet --upgrade pip
!pip install --quiet --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" unsloth-zoo || true
!pip install --quiet transformers trl peft accelerate bitsandbytes
!pip install --quiet fastapi uvicorn httpx pydantic statsmodels scipy scikit-learn tabulate

import os
import sys
from pathlib import Path

# Verify and set working directory to repository root (Supports Kaggle, Colab, Local)
if os.path.exists("/kaggle/working"):
    repo_dir = "/kaggle/working/vericall"
    if not os.path.exists(repo_dir):
        os.system(f"git clone https://github.com/bhuvneshsahu01/vericall.git {repo_dir}")
    os.chdir(repo_dir)
elif os.path.exists("/content"):
    repo_dir = "/content/vericall"
    if not os.path.exists(repo_dir):
        os.system(f"git clone https://github.com/bhuvneshsahu01/vericall.git {repo_dir}")
    os.chdir(repo_dir)
else:
    curr = Path.cwd()
    if (curr / "pyproject.toml").exists():
        print(f"[✓] Already at repository root: {curr}")
    elif (curr.parent / "pyproject.toml").exists():
        os.chdir(str(curr.parent))
    else:
        print(f"[!] Current directory: {curr}")

sys.path.insert(0, os.getcwd())
print("[✓] Environment ready. Working directory:", os.getcwd())


### Stage 2: In-Process FastAPI Mock Sandbox & Validator


In [ ]:
from vericall.sandbox.validator import ToolCallValidator
from vericall.sandbox.app import app

validator = ToolCallValidator()
print(f"[✓] Initialized ToolCallValidator with {len(validator.registry)} endpoints across 4 domains.")

# Smoke test all 12 endpoints
test_endpoints = [
    ("get_stock_quote", {"symbol": "AAPL", "exchange": "NASDAQ"}),
    ("execute_trade", {"account_id": "acc_1001", "symbol": "AAPL", "quantity": 10, "order_type": "market"}),
    ("get_account_balance", {"account_id": "acc_1001"}),
    ("search_products", {"query": "monitor", "category": "electronics"}),
    ("create_order", {"customer_id": "cust_101", "shipping_city": "Austin", "postal_code": "78701", "items": [{"product_id": "prod_1", "quantity": 1}]}),
    ("track_shipment", {"tracking_number": "TRK12345678", "carrier": "fedex"}),
    ("provision_server", {"instance_type": "c5.xlarge", "region": "us-east-1", "count": 1, "environment": "prod"}),
    ("restart_service", {"service_name": "nginx", "cluster_id": "k8s-prod-us-east-1"}),
    ("get_cluster_metrics", {"cluster_id": "k8s-prod-us-east-1", "metric_name": "cpu_utilization", "time_window_minutes": 15}),
    ("search_flights", {"origin_airport": "JFK", "destination_airport": "LHR", "departure_date": "2026-11-01", "passengers": 1, "cabin_class": "economy"}),
    ("book_hotel", {"hotel_id": "htl-101", "checkin_date": "2026-12-01", "checkout_date": "2026-12-05", "guests": 1, "room_type": "standard"}),
    ("cancel_booking", {"booking_id": "bk-12345", "reason": "schedule_change"}),
]

for fn, args in test_endpoints:
    res = validator.validate_call(fn, args)
    assert res.is_success, f"Endpoint {fn} failed: {res.error_message}"

print("[✓] All 12 sandbox endpoints verified functional.")


### Stage 3: Load Model & Tokenizer (Dual Engine with Automatic Fallback)


In [ ]:
import torch

max_seq_length = 2048
model_id = "Qwen/Qwen2.5-7B-Instruct"
USE_UNSLOTH = False

# Try Unsloth first
try:
    from unsloth import FastLanguageModel
    print(f"[*] Loading {model_id} via Unsloth FastLanguageModel (4-bit NF4)...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen2.5-7B-Instruct",
        max_seq_length=max_seq_length,
        load_in_4bit=True,
    )
    USE_UNSLOTH = True
    print("[✓] Loaded successfully via Unsloth engine.")
except Exception as unsloth_err:
    print(f"[!] Unsloth unavailable ({unsloth_err}).")
    print(f"[*] Activating HuggingFace Transformers + BitsAndBytes (4-bit NF4) fallback engine...")

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    print("[✓] Loaded successfully via HuggingFace Transformers engine.")

vram_gb = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print(f"[✓] Model ready on device. VRAM allocated: {vram_gb:.2f} GB")


### Stage 4: Evaluate Base Model on Frozen M=500 Unseen APIs


In [ ]:
import json
from pathlib import Path
from tqdm import tqdm
from vericall.eval.evaluate import evaluate_predictions

# Load M=500 unseen API test set
unseen_path = Path("data/eval_unseen_apis.jsonl")
test_records = [json.loads(line) for line in open(unseen_path, "r", encoding="utf-8") if line.strip()]
print(f"[*] Loaded {len(test_records)} frozen unseen API evaluation requests.")

TOOL_SYSTEM_PROMPT = """You are a precise, tool-calling assistant. Given the user's request, produce a JSON object representing the exact tool call with the required parameters.

Output format:
{"name": "<function_name>", "arguments": {<key_value_pairs>}}
"""

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
else:
    model.eval()

device = next(model.parameters()).device
base_predictions = []
print("[*] Generating predictions with Base Model...")
for item in tqdm(test_records):
    prompt = f"<|im_start|>system\n{TOOL_SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{item['query']}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True, temperature=0.1)
    response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    try:
        parsed = json.loads(response_text)
        fn = parsed.get("name", "")
        args = parsed.get("arguments", {})
    except Exception:
        fn = "invalid_json"
        args = {}

    base_predictions.append({"name": fn, "arguments": args})

base_results = evaluate_predictions(test_records, base_predictions, "base_model")
print(f"\n[Base Model] Unseen Accuracy: {base_results['accuracy_pct']}% (Passed: {base_results['passed_requests']}/{base_results['total_requests']})")
print(f"             95% Bootstrap CI: [{base_results['bootstrap_95_ci'][0]}%, {base_results['bootstrap_95_ci'][1]}%]")
print(f"             Errors: {base_results['error_taxonomy']}")


### Stage 5: QLoRA Supervised Fine-Tuning (SFT)


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset
import inspect
import torch

try:
    from trl import SFTConfig
    HAS_SFT_CONFIG = True
except ImportError:
    HAS_SFT_CONFIG = False

from vericall.train.train_sft import load_sft_dataset

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
else:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    if torch.cuda.is_available():
        model = prepare_model_for_kbit_training(model)
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_config)

# Ensure trainable adapter parameters are in float32 for AdamW stability
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

sft_data = load_sft_dataset()
train_dataset = Dataset.from_list(sft_data)

# Safe subclass to bypass experimental fused_lm_head slicing bug in newer TRL
class SafeSFTTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        kwargs = {}
        if num_items_in_batch is not None:
            kwargs["num_items_in_batch"] = num_items_in_batch
        return super(SFTTrainer, self).compute_loss(model, inputs, return_outputs=return_outputs, **kwargs)

common_args = {
    "output_dir": "checkpoints/runs_sft",
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "warmup_steps": 10,
    "num_train_epochs": 1,
    "learning_rate": 2e-4,
    "fp16": False,
    "bf16": False,
    "logging_steps": 20,
    "optim": "adamw_torch",
    "weight_decay": 0.01,
    "lr_scheduler_type": "cosine",
    "seed": 42,
    "report_to": "none",
}

sft_kwargs = {
    "model": model,
    "train_dataset": train_dataset,
}

if HAS_SFT_CONFIG:
    sft_cfg_params = inspect.signature(SFTConfig.__init__).parameters
    cfg_kwargs = dict(common_args)
    if "dataset_text_field" in sft_cfg_params:
        cfg_kwargs["dataset_text_field"] = "text"
    if "max_length" in sft_cfg_params:
        cfg_kwargs["max_length"] = max_seq_length
    elif "max_seq_length" in sft_cfg_params:
        cfg_kwargs["max_seq_length"] = max_seq_length
    training_args = SFTConfig(**cfg_kwargs)
else:
    training_args = TrainingArguments(**common_args)
    sft_kwargs["dataset_text_field"] = "text"

sft_kwargs["args"] = training_args

sig = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in sig:
    sft_kwargs["processing_class"] = tokenizer
else:
    sft_kwargs["tokenizer"] = tokenizer
if "max_seq_length" in sig and not HAS_SFT_CONFIG:
    sft_kwargs["max_seq_length"] = max_seq_length

trainer = SafeSFTTrainer(**sft_kwargs)

print("[*] Starting QLoRA SFT training...")
trainer.train()

model.save_pretrained("checkpoints/vericall-sft-lora")
tokenizer.save_pretrained("checkpoints/vericall-sft-lora")
print("[✓] SFT training complete. Adapters saved to checkpoints/vericall-sft-lora.")


### Stage 6: Evaluate SFT Model on M=500 Unseen APIs


In [ ]:
if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
else:
    model.eval()

device = next(model.parameters()).device
sft_predictions = []
print("[*] Generating predictions with SFT Model...")
for item in tqdm(test_records):
    prompt = f"<|im_start|>system\n{TOOL_SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{item['query']}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True, temperature=0.1)
    response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    try:
        parsed = json.loads(response_text)
        fn = parsed.get("name", "")
        args = parsed.get("arguments", {})
    except Exception:
        fn = "invalid_json"
        args = {}

    sft_predictions.append({"name": fn, "arguments": args})

sft_results = evaluate_predictions(test_records, sft_predictions, "sft_only")
print(f"\n[SFT Model] Unseen Accuracy: {sft_results['accuracy_pct']}% (Passed: {sft_results['passed_requests']}/{sft_results['total_requests']})")
print(f"            95% Bootstrap CI: [{sft_results['bootstrap_95_ci'][0]}%, {sft_results['bootstrap_95_ci'][1]}%]")
print(f"            Errors: {sft_results['error_taxonomy']}")


### Stage 7: Execution-Grounded DPO Near-Miss Mining


In [ ]:
from vericall.data.mine_dpo import mine_dpo_pairs

dpo_pairs = mine_dpo_pairs(num_pairs=1000)
print(f"[✓] Successfully mined {len(dpo_pairs)} execution-verified preference pairs.")
print("Sample Pair:")
print("  Prompt:  ", dpo_pairs[0]["prompt"])
print("  Chosen:  ", dpo_pairs[0]["chosen"])
print("  Rejected:", dpo_pairs[0]["rejected"], f"({dpo_pairs[0]['rejected_status']})")


### Stage 8: QLoRA Direct Preference Optimization (DPO)


In [ ]:
import inspect
import types
import torch
from datasets import Dataset
from transformers.modeling_outputs import CausalLMOutputWithPast
from trl import DPOTrainer, DPOConfig
from vericall.train.train_dpo import load_dpo_dataset

if USE_UNSLOTH:
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()

# 1. Neutralize TRL's internal fused_lm_head monkey-patcher
try:
    import trl.trainer.utils
    trl.trainer.utils.add_fused_lm_head = lambda *args, **kwargs: None
except Exception:
    pass

# 2. Universal fallback for CausalLMOutputWithPast so unexpected metric queries never crash
def safe_output_getattr(self, name):
    if hasattr(self, "logits") and self.logits is not None:
        return torch.zeros(
            (self.logits.shape[0], self.logits.shape[1] - 1),
            device=self.logits.device,
        )
    raise AttributeError(f"'{type(self).__name__}' object has no attribute '{name}'")

CausalLMOutputWithPast.__getattr__ = safe_output_getattr

# 3. Revert model.forward from any instance patch back to clean PEFT class method
if "forward" in model.__dict__:
    del model.__dict__["forward"]

true_class_forward = model.__class__.forward

# 4. Comprehensive forward wrapper that provides all DPO metrics
def safe_forward(self, *args, **kwargs):
    kwargs.pop("fused_lm_head", None)
    labels = kwargs.get("labels", None)
    shift_labels = kwargs.pop("shift_labels", None)
    kwargs.pop("num_items_in_batch", None)
    kwargs.pop("support_ids", None)

    outputs = true_class_forward(self, *args, **kwargs)

    if hasattr(outputs, "logits") and outputs.logits is not None:
        logits = outputs.logits
        shift_logits = logits[:, :-1, :].contiguous()

        with torch.no_grad():
            probs = shift_logits.softmax(dim=-1)
            log_probs_full = shift_logits.log_softmax(dim=-1)
            outputs.entropy = -(probs * log_probs_full).sum(dim=-1)
            outputs.mean_logits = shift_logits.mean(dim=-1)
            outputs.log_sum_sq_probs = (probs ** 2).sum(dim=-1).log()

        target_labels = shift_labels if shift_labels is not None else labels
        if target_labels is not None:
            shift_lbls = target_labels[:, 1:].contiguous() if shift_labels is None else target_labels.contiguous()
            if shift_lbls.shape[1] == shift_logits.shape[1]:
                loss_mask = shift_lbls != -100
                safe_lbls = shift_lbls.clone()
                safe_lbls[~loss_mask] = 0

                per_token_logps = torch.gather(
                    shift_logits.log_softmax(-1), dim=-1, index=safe_lbls.unsqueeze(-1)
                ).squeeze(-1)
                outputs.log_probs = torch.where(loss_mask, per_token_logps, torch.zeros_like(per_token_logps))

                with torch.no_grad():
                    outputs.is_top1 = (shift_logits.argmax(dim=-1) == safe_lbls) & loss_mask
            else:
                outputs.is_top1 = torch.zeros(shift_logits.shape[:2], dtype=torch.bool, device=shift_logits.device)
                outputs.log_probs = torch.zeros(shift_logits.shape[:2], dtype=shift_logits.dtype, device=shift_logits.device)
        else:
            outputs.is_top1 = torch.zeros(shift_logits.shape[:2], dtype=torch.bool, device=shift_logits.device)
            outputs.log_probs = torch.zeros(shift_logits.shape[:2], dtype=shift_logits.dtype, device=shift_logits.device)

    return outputs

model.forward = types.MethodType(safe_forward, model)
model.train()

# 5. Keep trainable adapter parameters in float32 for optimizer stability
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

# 6. Configure DPO
cfg_params = inspect.signature(DPOConfig.__init__).parameters
dpo_cfg_kwargs = {
    "output_dir": "checkpoints/runs_dpo",
    "beta": 0.1,
    "learning_rate": 5e-5,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "num_train_epochs": 1,
    "optim": "adamw_torch",
    "warmup_ratio": 0.1,
    "lr_scheduler_type": "cosine",
    "logging_steps": 10,
    "seed": 42,
    "fp16": False,
    "bf16": False,
    "report_to": "none",
}

if "max_length" in cfg_params:
    dpo_cfg_kwargs["max_length"] = max_seq_length
if "max_prompt_length" in cfg_params:
    dpo_cfg_kwargs["max_prompt_length"] = 1024

dpo_config = DPOConfig(**dpo_cfg_kwargs)

dpo_pairs_data = load_dpo_dataset()
dpo_train_dataset = Dataset.from_list(dpo_pairs_data)

dpo_kwargs = {
    "model": model,
    "ref_model": None,
    "args": dpo_config,
    "train_dataset": dpo_train_dataset,
}

sig_dpo = inspect.signature(DPOTrainer.__init__).parameters
if "processing_class" in sig_dpo:
    dpo_kwargs["processing_class"] = tokenizer
else:
    dpo_kwargs["tokenizer"] = tokenizer

dpo_trainer = DPOTrainer(**dpo_kwargs)

print("[*] Starting QLoRA DPO alignment...")
dpo_trainer.train()

model.save_pretrained("checkpoints/vericall-dpo-lora")
tokenizer.save_pretrained("checkpoints/vericall-dpo-lora")
print("[✓] DPO training complete. Adapters saved to checkpoints/vericall-dpo-lora.")


### Stage 9: Final Evaluation & Statistical Rigor (Bootstrap CI & McNemar Test)


In [ ]:
from vericall.eval.stats import compute_mcnemar_test

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
else:
    model.eval()

device = next(model.parameters()).device
dpo_predictions = []
print("[*] Generating final predictions with VeriCall (SFT+DPO)...")
for item in tqdm(test_records):
    prompt = f"<|im_start|>system\n{TOOL_SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{item['query']}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True, temperature=0.1)
    response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    try:
        parsed = json.loads(response_text)
        fn = parsed.get("name", "")
        args = parsed.get("arguments", {})
    except Exception:
        fn = "invalid_json"
        args = {}

    dpo_predictions.append({"name": fn, "arguments": args})

dpo_results = evaluate_predictions(test_records, dpo_predictions, "sft_dpo_vericall")

# Compute McNemar tests
stat_base, p_base = compute_mcnemar_test(base_results["binary_scores"], dpo_results["binary_scores"])
stat_sft, p_sft = compute_mcnemar_test(sft_results["binary_scores"], dpo_results["binary_scores"])

print(f"\n=======================================================")
print(f"FINAL VERICALL BENCHMARK RESULTS (M=500 Unseen APIs)")
print(f"=======================================================")
print(f"[Base Model]     Accuracy: {base_results['accuracy_pct']}%  95% CI: [{base_results['bootstrap_95_ci'][0]}%, {base_results['bootstrap_95_ci'][1]}%]")
print(f"[SFT-Only]       Accuracy: {sft_results['accuracy_pct']}%  95% CI: [{sft_results['bootstrap_95_ci'][0]}%, {sft_results['bootstrap_95_ci'][1]}%]")
print(f"[VeriCall (DPO)] Accuracy: {dpo_results['accuracy_pct']}%  95% CI: [{dpo_results['bootstrap_95_ci'][0]}%, {dpo_results['bootstrap_95_ci'][1]}%]")
print(f"\nMcNemar Test vs Base:    chi2 = {stat_base:.4f}, p = {p_base:.4e} (p < 0.05: {p_base < 0.05})")
print(f"McNemar Test vs SFT-Only: chi2 = {stat_sft:.4f}, p = {p_sft:.4e} (p < 0.05: {p_sft < 0.05})")
print(f"=======================================================")


### Stage 10: Save Evaluation Summary & Render Results


In [ ]:
from pathlib import Path
import json
import os
import shutil

summary_data = {
    "metadata": {
        "model_base": model_id,
        "eval_set": "eval_unseen_apis.jsonl",
        "eval_count": len(test_records),
        "mode": "full_benchmark",
    },
    "configs": {
        "base_model": base_results,
        "sft_only": sft_results,
        "sft_dpo_vericall": dpo_results,
    },
    "mcnemar_tests": {
        "vericall_vs_base": {"statistic": stat_base, "p_value": p_base},
        "vericall_vs_sft": {"statistic": stat_sft, "p_value": p_sft},
    }
}

Path("results").mkdir(exist_ok=True)
with open("results/eval_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=2)

print("[✓] Saved evaluation summary to results/eval_summary.json")

# If on Kaggle, copy to top-level output directory for 1-click download
if os.path.exists("/kaggle/working"):
    shutil.copy2("results/eval_summary.json", "/kaggle/working/eval_summary.json")
    print("[✓] Copied to /kaggle/working/eval_summary.json for direct artifact download")

!python scripts/render_results.py
